<a href="https://colab.research.google.com/github/RuleOfNight/dialog-sumarizer/blob/main/SummaryPaper_(2)_(1)_(2)_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers peft pandas spacy datasets rouge_score

  Preparing metadata (setup.py) ... done


In [ ]:
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 39.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import os
import torch
import spacy
import numpy as np
import pandas as pd
import random
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from rouge_score import rouge_scorer

from transformers.modeling_outputs import BaseModelOutput
from transformers import AutoTokenizer, BartForConditionalGeneration, BartConfig

from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Tokenizer & add Special Tokens
tokenizer = AutoTokenizer.from_pretrained('facebook/bart-base')

# Thêm Special Tokens
special_tokens_dict = {'additional_special_tokens': ['#Person1#', '#Person2#']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
print(f"✅ Tokenizer & add Special Tokens: Checked")

raw_dataset = load_dataset("knkarthick/samsum")
print("✅ Dataset checked")

Device: cuda


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

✅ Dataset checked


In [ ]:
# DATASET CLASS VỚI GRAPH ALIGNMENT
class GraphDialogSumDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_input_len=512, max_target_len=128):
        # Nhận tokenizer đã update từ bên ngoài vào
        self.data = hf_dataset
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

        # Load Spacy
        self.nlp = spacy.load("en_core_web_sm", disable=['ner'])

        # TẠO TỪ ĐIỂN DEPENDENCY RELATION (EDGE TYPES)
        self.dep_labels = self.nlp.get_pipe("parser").labels
        self.dep2id = {label: i+1 for i, label in enumerate(self.dep_labels)}
        self.dep2id['UNK'] = len(self.dep2id) + 1
        self.dep2id['SELF'] = len(self.dep2id) + 1
        self.dep2id['PAD'] = 0
        print(f"Vocab Dependency (Edge Types): {len(self.dep2id)} relations")

        # Cache Speaker IDs (để nhất quán ID người nói trong 1 đoạn hội thoại)
        self.global_speaker_id_map = {}

    def __len__(self):
        return len(self.data)

        # Mapping cho đoạn dialog
        # Cơ bản thì cái map có speaker_ids và turn_ids nó trông ntn
        #              Alice: Are you free tmr? \nBob: Yes, but not for you \n Alice: Oh okay
        # speaker_ids:
        # turn_ids:
        # and so on...

    # Vấn đề là nếu tính theo token của Spacy thì dễ rồi, nhưng token của BART là theo subword
    # 1 từ mà chia ra nhiều subword thì khó cho model gán được turn_id
    def _parse_dialogue_structure(self, text):
        """
        Phân tích cấu trúc hội thoại:
        Input: "Alice: Hi there\nBob: Hello"
        Output: List các đoạn meta [(start_char, end_char, speaker_name, turn_id)]
        """
        lines = text.split('\n')
        meta_segments = []
        current_char_idx = 0

        # Mapping cục bộ cho mỗi đoạn hội thoại (Alice -> 1, Bob -> 2)
        local_speaker_map = {}
        next_speaker_id = 1

        for turn_id, line in enumerate(lines):
            # 1. Xác định người nói dựa trên format "Name: Text"
            if ':' in line:
                speaker_name, content = line.split(':', 1)
                speaker_name = speaker_name.strip()
            else:
                speaker_name = "UNK"

            # 2. Gán ID cho người nói
            if speaker_name not in local_speaker_map:
                local_speaker_map[speaker_name] = next_speaker_id
                next_speaker_id += 1

            spk_id = local_speaker_map[speaker_name]

            # 3. Lưu metadata vị trí ký tự
            # +1 cho ký tự \n đã bị split mất (để khớp với len của raw text không tí nó lỗi shape)
            end_char_idx = current_char_idx + len(line) + 1

            meta_segments.append({
                'start': current_char_idx,
                'end': end_char_idx,
                'spk_id': spk_id,
                'turn_id': turn_id + 1 # Turn bắt đầu từ 1
            })

            current_char_idx = end_char_idx
        return meta_segments


    # hàm trả về 1 sample cho mỗi lần Dataloader yêu cầu (trong này xử lí khác biệt về token của spacy với bart)
    def __getitem__(self, idx):
        item = self.data[idx]
        dialogue = item['dialogue']
        summary = item['summary']

        # Bước 1L Lấy thông tin vị trí của từng lượt thoại
        structure_meta = self._parse_dialogue_structure(dialogue)

        # Bước 2: spacy prasing + tokenization
        # Cắt ngắn bớt để tránh OOM spaCy với văn bản quá dài
        doc = self.nlp(dialogue[:5000])

        # Khởi tạo mảng chứa
        subword_tokens = [self.tokenizer.bos_token_id]
        speaker_ids_list = [0]
        turn_ids_list = [0]

        spacy_to_subword_map = {}

        # chia spacy tokes ra thành các subword tokens của bart và meta data
        for spacy_token in doc:
            prefix = " " if spacy_token.i > 0 else ""
            # chia token spacy thành dạng subword
            new_sub_tokens = self.tokenizer.encode(prefix + spacy_token.text, add_special_tokens=False)

            if not new_sub_tokens:
                continue

            # Cần để ý giữ chỗ cho EOS và BOS tokens
            # Cần 1 slot cho EOS:  self.max_input_len - len(subword_tokens) - 1
            available_space_for_word = self.max_input_len - len(subword_tokens) - 1

            if available_space_for_word <= 0: # No space for current word or EOS
                break

            if len(new_sub_tokens) > available_space_for_word:
                new_sub_tokens = new_sub_tokens[:available_space_for_word] # Truncate current word's sub_tokens

            if not new_sub_tokens: # If after truncation, no tokens left from this word
                break

            # speaker_ids và turn_ids cho token Spacy hiện tại
            curr_spk_id = 0
            curr_turn_id = 0
            token_start_char = spacy_token.idx
            for seg in structure_meta:
                if seg['start'] <= token_start_char < seg['end']:
                    curr_spk_id = seg['spk_id']
                    curr_turn_id = seg['turn_id']
                    break

            # Map từ thứ spacy_token.i khi dịch sang subword thì bắt đầu ở vị trí thứ subword_tokens của mảng input
            spacy_to_subword_map[spacy_token.i] = len(subword_tokens)

            # dupicate speaker_id và turn_id để gán cho các subword mới tách
            subword_tokens.extend(new_sub_tokens)
            speaker_ids_list.extend([curr_spk_id] * len(new_sub_tokens))
            turn_ids_list.extend([curr_turn_id] * len(new_sub_tokens))

        # Add EOS Token
        subword_tokens.append(self.tokenizer.eos_token_id)
        speaker_ids_list.append(0)
        turn_ids_list.append(0)

        seq_len = len(subword_tokens) # This should now be <= self.max_input_len

        # Safeguard: Explicitly truncate if seq_len exceeded max_input_len due to any unforeseen edge case
        if seq_len > self.max_input_len:
            # print(f"DEBUG: Force truncating sample {idx}. Original seq_len: {seq_len}, max_input_len: {self.max_input_len}") # For debugging
            subword_tokens = subword_tokens[:self.max_input_len]
            speaker_ids_list = speaker_ids_list[:self.max_input_len]
            turn_ids_list = turn_ids_list[:self.max_input_len]
            seq_len = self.max_input_len # Update seq_len after force truncation


        # Bước 3: Tạo denpency graph
        # tạo ra matrix kích thước max_input_len x max_input_len. Thể hiện từ nào bổ nghĩa từ nào
        dep_matrix = torch.zeros((self.max_input_len, self.max_input_len), dtype=torch.long)

        for token in doc:
            # Only consider tokens that were included in the truncated sequence
            if token.i in spacy_to_subword_map:
                child_start_idx = spacy_to_subword_map[token.i]
                head_start_idx = spacy_to_subword_map.get(token.head.i, -1) # Use .get with default -1 for safety

                # Only add edges for tokens that are within the actual sequence length (after potential truncation)
                if child_start_idx < seq_len:
                    # self edge
                    dep_matrix[child_start_idx, child_start_idx] = self.dep2id['SELF']

                    if head_start_idx != -1 and head_start_idx < seq_len: # Add edge to head if head also exists in sequence
                        rel_id = self.dep2id.get(token.dep_, self.dep2id['UNK'])
                        dep_matrix[child_start_idx, head_start_idx] = rel_id

        # bởi vì do bước convert token spacy qua subword ở trên, nên các từ bị tách ra sẽ chỉ được nối
        # 1 mũi tên quan hệ ngữ pháp vào từ đầu tiên của nó, ví dụ playing tách thành play và ing thì
        # các mũi tên quan hệ sẽ chỉ nối tới từ play, bỏ qua ing (toàn bộ hàng/cột của subword ing sẽ full 0)
        # Transformer có attention nên nó sẽ tự hiểu play đi với ing, không cần lo lắm


        # Bước 4: Padding
        pad_len = self.max_input_len - seq_len

        # Pad Input IDs
        input_ids = subword_tokens + [self.tokenizer.pad_token_id] * pad_len
        attention_mask = [1] * seq_len + [0] * pad_len

        # Pad Speaker & Turn (Dùng 0 để padding)
        speaker_ids = speaker_ids_list + [0] * pad_len
        turn_ids = turn_ids_list + [0] * pad_len

        # Bước 5: xử lí output
        labels = self.tokenizer(
            text_target=summary,
            max_length=self.max_target_len,
            truncation=True,
            padding="max_length"
        )
        # trick lỏ: thay mấy cái token PAD từ 1 thành -100 sẽ giúp model skip mấy token này
        labels_ids = [l if l != self.tokenizer.pad_token_id else -100 for l in labels['input_ids']]

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'dep_matrix': dep_matrix,
            'labels': torch.tensor(labels_ids, dtype=torch.long),
            'speaker_ids': torch.tensor(speaker_ids, dtype=torch.long),
            'turn_ids': torch.tensor(turn_ids, dtype=torch.long)
        }

In [ ]:
BATCH_SIZE = 16

train_ds = GraphDialogSumDataset(raw_dataset['train'], tokenizer)
val_ds = GraphDialogSumDataset(raw_dataset['validation'], tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("✅ DataLoader checked")

sample = train_ds[0]
print("Shape Input:", sample['input_ids'].shape)       # torch.Size([512])
print("Shape Speaker:", sample['speaker_ids'].shape)   # torch.Size([512]) -> Phải khớp Input
print("Shape Turn:", sample['turn_ids'].shape)         # torch.Size([512]) -> Phải khớp Input
print("Speaker Sample:", sample['speaker_ids'][:40])

Vocab Dependency (Edge Types): 48 relations
Vocab Dependency (Edge Types): 48 relations
✅ DataLoader checked
Shape Input: torch.Size([512])
Shape Speaker: torch.Size([512])
Shape Turn: torch.Size([512])
Speaker Sample: tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0])


In [ ]:
class EdgeBiasGenerator(nn.Module):
    def __init__(self, num_rels, num_heads):
        super().__init__()
        self.edge_embedding = nn.Embedding(num_rels, num_heads)
        nn.init.normal_(self.edge_embedding.weight, mean=0.0, std=0.02)

    def forward(self, dep_matrix):
        bias = self.edge_embedding(dep_matrix)
        return bias.permute(0, 3, 1, 2)

In [ ]:
class GraphTransformerLayer(nn.Module):
    def __init__(self, hidden_dim=768, num_heads=12, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim)

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim * 4, hidden_dim)
        )

        # zero init
        nn.init.zeros_(self.o_proj.weight)
        nn.init.zeros_(self.o_proj.bias)

    def forward(self, x, edge_bias, attention_mask=None):
        batch_size, seq_len, _ = x.shape
        residual = x
        x = self.norm1(x)

        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn_scores = attn_scores + edge_bias # Inject Graph Bias

        # cho attention score của padding về -inf
        if attention_mask is not None:
            extended_mask = (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -1e9
            attn_scores = attn_scores + extended_mask

        attn_probs = self.dropout(F.softmax(attn_scores, dim=-1))
        context = torch.matmul(attn_probs, V).transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        output = self.o_proj(context)
        x = residual + self.dropout(output)
        x = x + self.ffn(self.norm2(x))
        return x

In [ ]:
class GraphBARTModel(nn.Module):
    def __init__(self, model_name='facebook/bart-base', num_dep_rels=50, num_graph_layers=2, max_turn_id=100, num_speakers=20):
        super().__init__()
        print(f"Loading Backbone {model_name}...")
        self.bart = BartForConditionalGeneration.from_pretrained(model_name)

        # Resize embeddings để khớp với tokenizer sau khi thêm special tokens
        # Lưu ý: Cần truyền tokenizer len từ bên ngoài vào hoặc resize sau khi khởi tạo
        # Ở đây giữ logic cũ, giả sử resize sẽ được gọi bên ngoài hoặc trong main

        config = self.bart.config
        self.d_model = config.d_model

        # embedding matrix cho speaker và turn thoại
        self.speaker_embed = nn.Embedding(num_speakers, self.d_model)
        self.turn_embed = nn.Embedding(max_turn_id, self.d_model)

        # Init weights: Sử dụng phân phối chuẩn giống BART để hội tụ tốt hơn
        nn.init.normal_(self.speaker_embed.weight, mean=0.0, std=config.init_std)
        nn.init.normal_(self.turn_embed.weight, mean=0.0, std=config.init_std)

        # Graph Layers cũ
        self.edge_bias_gen = EdgeBiasGenerator(num_dep_rels, config.encoder_attention_heads)
        self.graph_layers = nn.ModuleList([
            GraphTransformerLayer(config.d_model, config.encoder_attention_heads)
            for _ in range(num_graph_layers)
        ])

    def _prepare_inputs_embeds(self, input_ids, speaker_ids, turn_ids):
        """Hàm cộng gộp các vector"""


        # BART scale embeddings bằng sqrt(d_model)
        # Nếu train có kq tệ thì thử cho inputs_embeds x d_model**0.5 tại sợ 2 edge speaker và turn bị át mất
        inputs_embeds = self.bart.model.encoder.embed_tokens(input_ids)

        if speaker_ids is not None:
            inputs_embeds += self.speaker_embed(speaker_ids)
        if turn_ids is not None:
            inputs_embeds += self.turn_embed(turn_ids)

        return inputs_embeds

    def forward(self, input_ids, attention_mask, dep_matrix, speaker_ids=None, turn_ids=None, labels=None):
        # Inject Embeddings vào Encoder
        # Tạo inputs_embeds tổng hợp
        inputs_embeds = self._prepare_inputs_embeds(input_ids, speaker_ids, turn_ids)

        encoder_outputs = self.bart.model.encoder(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask
        )

        hidden_state = encoder_outputs.last_hidden_state

        # Graph Processing
        edge_bias = self.edge_bias_gen(dep_matrix)
        for layer in self.graph_layers:
            hidden_state = layer(hidden_state, edge_bias, attention_mask)

        # Decoder
        outputs = self.bart(
            encoder_outputs=(hidden_state,),
            attention_mask=attention_mask,
            labels=labels
        )
        return outputs

    def generate(self, input_ids, attention_mask, dep_matrix, speaker_ids=None, turn_ids=None, **kwargs):
        # Cập nhật Generate cho Inference
        # Tương tự forward, cần embedding tổng hợp trước khi encode
        inputs_embeds = self._prepare_inputs_embeds(input_ids, speaker_ids, turn_ids)

        encoder_outputs = self.bart.model.encoder(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask
        )

        hidden_state = encoder_outputs.last_hidden_state

        # Graph Processing
        edge_bias = self.edge_bias_gen(dep_matrix)
        for layer in self.graph_layers:
            hidden_state = layer(hidden_state, edge_bias, attention_mask)

        # Gói lại thành object để truyền vào hàm generate của BART
        encoded_output_obj = BaseModelOutput(last_hidden_state=hidden_state)

        return self.bart.generate(
            encoder_outputs=encoded_output_obj,
            attention_mask=attention_mask,
            **kwargs
        )

NUM_RELS = len(train_ds.dep2id) + 5
model = GraphBARTModel(num_dep_rels=NUM_RELS).to(device)
print("✅ BART checked")

Loading Backbone facebook/bart-base...


model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

✅ BART checked


In [ ]:
def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'],
                                      use_stemmer=True) # Chuyển các từ về dạng nguyên thể của nó (vd playing và plays đều về play)
    agg_scores = {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        agg_scores['rouge1'] += scores['rouge1'].fmeasure
        agg_scores['rouge2'] += scores['rouge2'].fmeasure
        agg_scores['rougeL'] += scores['rougeL'].fmeasure
    n = len(predictions)
    return {k: v/n for k, v in agg_scores.items()}

def freeze_backbone(model):
    print("\n Freeze backbone")
    for param in model.bart.parameters():
        param.requires_grad = False

    for param in model.edge_bias_gen.parameters():
        param.requires_grad = True
    for param in model.graph_layers.parameters():
        param.requires_grad = True
    for param in model.speaker_embed.parameters():
        param.requires_grad = True
    for param in model.turn_embed.parameters():
        param.requires_grad = True

def unfreeze_all(model):
    print("\n🔓 UNFREEZE.")
    for param in model.parameters():
        param.requires_grad = True

print("✅ Init checked.")

✅ Init checked.


# Engines

In [ ]:
import torch.optim as optim

def train_engine(model, dataloader, optimizer, phase_name):
    model.train()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc=f"Training {phase_name}")

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        dep_matrix = batch['dep_matrix'].to(device)
        labels = batch['labels'].to(device)
        speaker_ids = batch['speaker_ids'].to(device)
        turn_ids = batch['turn_ids'].to(device)

        optimizer.zero_grad()

        # truyền params
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            dep_matrix=dep_matrix,
            speaker_ids=speaker_ids,
            turn_ids=turn_ids,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    return total_loss / len(dataloader)

def evaluate_engine(model, dataloader, tokenizer, mode="Val"):
    model.eval()
    total_loss = 0
    decoded_preds = []
    decoded_labels = []

    print(f"Đang đánh giá ({mode})...")
    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, desc="Evaluating")):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            dep_matrix = batch['dep_matrix'].to(device)
            labels = batch['labels'].to(device)
            speaker_ids = batch.get('speaker_ids')
            turn_ids = batch.get('turn_ids')

            if speaker_ids is not None: speaker_ids = speaker_ids.to(device)
            if turn_ids is not None: turn_ids = turn_ids.to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                dep_matrix=dep_matrix,
                speaker_ids=speaker_ids,
                turn_ids=turn_ids,
                labels=labels
            )

            if outputs.loss is None:
                if i == 0: print("Model trả về loss None")
                batch_loss = 0
            else:
                batch_loss = outputs.loss.item()

            total_loss += batch_loss

            # Thêm tham số chống lặp
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                dep_matrix=dep_matrix,
                speaker_ids=speaker_ids,
                turn_ids=turn_ids,
                max_length=128,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3,    # Cấm lặp cụm 3 từ
                repetition_penalty=1.2,    # Phạt nặng lặp từ
                length_penalty=1.0,
                decoder_start_token_id=tokenizer.bos_token_id
            )

            # Decode
            preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            labels_ids = labels.cpu().numpy()
            labels_ids = np.where(labels_ids != -100, labels_ids, tokenizer.pad_token_id)
            refs = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

            decoded_preds.extend(preds)
            decoded_labels.extend(refs)

    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0
    rouge = compute_rouge(decoded_preds, decoded_labels)
    print(f"   📉 {mode} Loss: {avg_loss:.4f}")
    print(f"   📝 ROUGE-1: {rouge['rouge1']:.4f} | ROUGE-L: {rouge['rougeL']:.4f}")
    return avg_loss, rouge

print("✅ Engines checked")

✅ Engines checked


# Main

In [ ]:
# Warm-up
print("\n GIAI ĐOẠN 1: WARM-UP...")
freeze_backbone(model)
warmup_params = filter(lambda p: p.requires_grad, model.parameters())

optimizer_warmup = optim.AdamW(warmup_params, lr=2e-4) # Giảm từ 1e-3 xuống 2e-4 tránh giao động mạnh

# Demo
train_loss = train_engine(model, train_loader, optimizer_warmup, "Warmup 1")
print(f"   Train Loss: {train_loss:.4f}")
evaluate_engine(model, val_loader, tokenizer, mode="Val (Warmup)")

# joint training
print("\n GIAI ĐOẠN 2: JOINT TRAINING...")
unfreeze_all(model)
backbone_params = [p for n, p in model.named_parameters() if "bart" in n]
graph_params = [p for n, p in model.named_parameters() if "bart" not in n]

optimizer_joint = optim.AdamW([
    {'params': backbone_params, 'lr': 2e-5},
    {'params': graph_params, 'lr': 1e-4}
])

# Demo 1 epoch (Joint)
train_loss = train_engine(model, train_loader, optimizer_joint, "Joint 1")
print(f"   Train Loss: {train_loss:.4f}")
evaluate_engine(model, val_loader, tokenizer, mode="Val Joint 1")


print("\n-- TEST:")
model.eval()
sample = next(iter(val_loader))
input_ids = sample['input_ids'][0].unsqueeze(0).to(device)
mask = sample['attention_mask'][0].unsqueeze(0).to(device)
dep = sample['dep_matrix'][0].unsqueeze(0).to(device)

speaker_ids = sample.get('speaker_ids')
turn_ids = sample.get('turn_ids')

speaker_ids = speaker_ids[0].unsqueeze(0).to(device)
turn_ids = turn_ids[0].unsqueeze(0).to(device)

# gen văn bản đầu ra
gen_ids = model.generate(
    input_ids=input_ids,
    attention_mask=mask,
    dep_matrix=dep,
    speaker_ids=speaker_ids,
    turn_ids=turn_ids,
    max_length=100,
    no_repeat_ngram_size=3,
    repetition_penalty=1.2,
    decoder_start_token_id=tokenizer.bos_token_id
)

print("Summary:", tokenizer.decode(gen_ids[0], skip_special_tokens=True))


 GIAI ĐOẠN 1: WARM-UP...

 Freeze backbone


Training Warmup 1:   0%|          | 0/921 [00:00<?, ?it/s]

   Train Loss: 2.7506
Đang đánh giá (Val (Warmup))...


Evaluating:   0%|          | 0/52 [00:00<?, ?it/s]

   📉 Val (Warmup) Loss: 2.0229
   📝 ROUGE-1: 0.2750 | ROUGE-L: 0.2428

 GIAI ĐOẠN 2: JOINT TRAINING...

🔓 UNFREEZE.


Training Joint 1:   0%|          | 0/921 [00:00<?, ?it/s]

   Train Loss: 2.0259
Đang đánh giá (Val Joint 1)...


Evaluating:   0%|          | 0/52 [00:00<?, ?it/s]

   📉 Val Joint 1 Loss: 1.7370
   📝 ROUGE-1: 0.4911 | ROUGE-L: 0.4051

-- TEST:
Summary: A wants to get a puppy for her son. B will take him to the animal shelter tomorrow afternoon. A will get him one of those little dogs called Lemmy.


# Continue training

In [ ]:
import os

MORE_EPOCHS = 3
START_EPOCH = 2  # Joint là 1 rồi nên phải start từ 2, từ 3 là...

# Biến lưu kq tốt nhất (chưa có từ bước trước thì init)
if 'best_rouge' not in locals():
    best_rouge = 0.0


for epoch in range(START_EPOCH, START_EPOCH + MORE_EPOCHS):
    print(f"\n--- Epoch {epoch} (Joint Training) ---")
    train_loss = train_engine(model, train_loader, optimizer_joint, f"Joint {epoch}")
    print(f"   Train Loss: {train_loss:.4f}")

    val_loss, val_rouge = evaluate_engine(model, val_loader, tokenizer, mode=f"Val {epoch}")

    # Checkpoint (lưu model nếu ROUGE-L tăng)
    current_rouge_l = val_rouge['rougeL']

    if current_rouge_l > best_rouge:
        improvement = current_rouge_l - best_rouge
        best_rouge = current_rouge_l

        # Lưu trọng số model
        save_path = "best_graph_bart_dialogsum.pt"
        torch.save(model.state_dict(), save_path)

        print(f"      Best model: ROUGE-L tăng: +{improvement:.4f} -> {best_rouge:.4f}")
    else:
        print(f"      Không cải thiện (Best: {best_rouge:.4f})")


model.eval()
random_idx = random.randint(0, len(val_ds)-1)
sample = val_ds[random_idx]

input_ids = sample['input_ids'].unsqueeze(0).to(device)
mask = sample['attention_mask'].unsqueeze(0).to(device)
dep = sample['dep_matrix'].unsqueeze(0).to(device)
# Fix: Correctly get speaker_ids and turn_ids from the current sample
speaker_ids = sample['speaker_ids'].unsqueeze(0).to(device)
turn_ids = sample['turn_ids'].unsqueeze(0).to(device)

gen_ids = model.generate(
        input_ids=input_ids, attention_mask=mask, dep_matrix=dep,
        speaker_ids=speaker_ids, turn_ids=turn_ids,
        max_length=128, no_repeat_ngram_size=3, repetition_penalty=1.2,
        decoder_start_token_id=tokenizer.bos_token_id
    )

print("Ground Truth:", tokenizer.decode(sample['labels'][sample['labels']!=-100], skip_special_tokens=True))
print("Generated   :", tokenizer.decode(gen_ids[0], skip_special_tokens=True))


--- Epoch 2 (Joint Training) ---


Training Joint 2:   0%|          | 0/921 [00:00<?, ?it/s]

   Train Loss: 1.7701
Đang đánh giá (Val 2)...


Evaluating:   0%|          | 0/52 [00:00<?, ?it/s]

   📉 Val 2 Loss: 1.6705
   📝 ROUGE-1: 0.4893 | ROUGE-L: 0.4087
      Best model: ROUGE-L tăng: +0.4087 -> 0.4087

--- Epoch 3 (Joint Training) ---


...

   📉 Val 3 Loss: 1.6723
   📝 ROUGE-1: 0.5070 | ROUGE-L: 0.4190
      Best model: ROUGE-L tăng: +0.0103 -> 0.4190

--- Epoch 4 (Joint Training) ---


Training Joint 4:   0%|          | 0/921 [00:00<?, ?it/s]

   Train Loss: 1.4827
Đang đánh giá (Val 4)...


Evaluating:   0%|          | 0/52 [00:00<?, ?it/s]

   📉 Val 4 Loss: 1.6535
   📝 ROUGE-1: 0.5082 | ROUGE-L: 0.4209
      Best model: ROUGE-L tăng: +0.0019 -> 0.4209
Ground Truth: IQ decreases by 20% after a 2-week holiday.
Generated   : George, Pete and Matt are wondering what happens after a gap year.


# Load best model & Inference test

In [ ]:
import random
import torch
import os


# Correct NUM_RELS to match the saved model (it was len(train_ds.dep2id) + 5 = 48 + 5 = 53 during training)
# The NUM_RELS in the current context is 55, which caused a mismatch.
# Using 53 as NUM_RELS to correctly load the edge_bias_gen.
NUM_RELS_SAVED = len(val_ds.dep2id) + 5 # This will be 48 + 5 = 53
model_best = GraphBARTModel(
    num_dep_rels=NUM_RELS_SAVED, # Use the correct number of relations
    num_speakers=20,
    max_turn_id=100
).to(device)

checkpoint_path = "best_graph_bart_dialogsum.pt"

if os.path.exists(checkpoint_path):
    print(f"Load model from: {checkpoint_path}")
    # Load state_dict BEFORE resizing token embeddings.
    # The saved model has a vocab size of 50265 (from checkpoint error message).
    # The current model_best.bart also has 50265 by default after initialization.
    model_best.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("Có model")
    # NOW, resize token embeddings to accommodate additional special tokens (50267 total).
    # This will add new embeddings for the two special tokens, initialized randomly,
    # while keeping the loaded weights for the original vocabulary.
    model_best.bart.resize_token_embeddings(len(tokenizer))
else:
    print("Không có model")
    model_best = model # Fallback nếu chưa lưu file

model_best.eval()

print("\n" + "="*50)
print(f" Test random 10 sample: ")
print("="*50)


random_indices = random.sample(range(len(val_ds)), 10)

with torch.no_grad():
    for i, idx in enumerate(random_indices):
        sample = val_ds[idx]

        input_ids = sample['input_ids'].unsqueeze(0).to(device)
        mask = sample['attention_mask'].unsqueeze(0).to(device)
        dep = sample['dep_matrix'].unsqueeze(0).to(device)
        # Fix for speaker_ids and turn_ids: These need to come from the 'sample' object for the current loop iteration
        speaker_ids = sample['speaker_ids'].unsqueeze(0).to(device)
        turn_ids = sample['turn_ids'].unsqueeze(0).to(device)

        # Lấy Ground Truth & Text gốc
        label_ids = sample['labels'].clone()
        label_ids[label_ids == -100] = tokenizer.pad_token_id
        ground_truth = tokenizer.decode(label_ids, skip_special_tokens=True)
        dialogue_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

        # Generate Summary
        gen_ids = model_best.generate(
            input_ids=input_ids,
            attention_mask=mask,
            dep_matrix=dep,
            speaker_ids=speaker_ids,
            turn_ids=turn_ids,
            max_length=128,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            length_penalty=1.0,
            decoder_start_token_id=tokenizer.bos_token_id
        )
        generated_summary = tokenizer.decode(gen_ids[0], skip_special_tokens=True)


        print(f"\n  SAMPLE {i+1} (Index {idx}):")
        print(f"    DIALOGUE: {dialogue_text[:200]}...")
        print(f"🎯 GROUND TRUTH: {ground_truth}")
        print(f"🤖 MODEL PREDICT: {generated_summary}")
        print("-" * 50)


Loading Backbone facebook/bart-base...


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Load model from: best_graph_bart_dialogsum.pt
Có model

 Test random 10 sample: 

  SAMPLE 1 (Index 114):
    DIALOGUE: Kyle : Who wants to go out for a drink ? 
 Megan : No , sorry , I 'm cleaning the house today . 
 Roseanne : You 've always loved cleaning , have n't you ? I remember how angry you used to get with yo...
🎯 GROUND TRUTH: Roseanne wanted to go out for a beer with Kyle and Vince, and leave her kids with Megan.  Megan will clean the house and give Roseanne's kids her famous chocolate cookies if they help, what made Roseanne change her mind, and she wants to join them.
🤖 MODEL PREDICT: Megan is cleaning the house today. She used to get angry with Vince's brother for leaving a mess in the kitchen. Kyle doesn't like cleaning. Roseanne prefers the floor.
--------------------------------------------------

  SAMPLE 2 (Index 25):
    DIALOGUE: Olivia : Hey Carter , are you still developing that restaurant business ? 
 Carter : Hi Olivia 
 Carter : Yes , we want to launch next m

In [ ]:
# Giải phóng hết vram trước rồi gọi model mới
import gc

# 1. Xóa các object model cũ nếu tồn tại
if 'model' in globals(): del model
if 'model_best' in globals(): del model_best
if 'optimizer_joint' in globals(): del optimizer_joint
if 'optimizer_warmup' in globals(): del optimizer_warmup

# 2. Xóa rác trong Python và Cache của PyTorch
gc.collect()
torch.cuda.empty_cache()

print(f"🧹 Đã dọn dẹp GPU. Bộ nhớ hiện tại: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

🧹 Đã dọn dẹp GPU. Bộ nhớ hiện tại: 4.15 GB


In [ ]:
checkpoint_path = "best_graph_bart_dialogsum.pt"
FINE_TUNE_EPOCHS = 3

NEW_LR_BACKBONE = 5e-7
NEW_LR_GRAPH = 2e-6

print(f" Load best model: {checkpoint_path}")

NUM_RELS = len(train_ds.dep2id) + 5
model_finetune = GraphBARTModel(
    num_dep_rels=NUM_RELS,
    num_speakers=20,
    max_turn_id=100
).to(device)


if os.path.exists(checkpoint_path):
    model_finetune.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("Có path")
else:
    raise FileNotFoundError("Không tìm thấy file")

model_finetune.bart.resize_token_embeddings(len(tokenizer))

# Tách tham số: 'bart' tên là backbone, còn lại (graph, speaker, turn) là nhóm mới
backbone_params = [p for n, p in model_finetune.named_parameters() if "bart" in n]
graph_params = [p for n, p in model_finetune.named_parameters() if "bart" not in n]

optimizer_finetune = optim.AdamW([
    {'params': backbone_params, 'lr': NEW_LR_BACKBONE},
    {'params': graph_params, 'lr': NEW_LR_GRAPH}
])


print("\n Evaluate baseline của model vừa load:")
_, val_rouge = evaluate_engine(model_finetune, val_loader, tokenizer, mode="Baseline")
best_rouge = val_rouge['rougeL']
print(f"      Started point: (Best ROUGE-L cũ): {best_rouge:.4f}")

print(f"\n Fine-tune {FINE_TUNE_EPOCHS} epochs")

for epoch in range(1, FINE_TUNE_EPOCHS + 1):
    print(f"\n--- Fine-tune Epoch {epoch}/{FINE_TUNE_EPOCHS} ---")

    # Train
    train_loss = train_engine(model_finetune, train_loader, optimizer_finetune, f"FT {epoch}")
    print(f"   Train Loss: {train_loss:.4f}")

    # Validate
    val_loss, val_rouge = evaluate_engine(model_finetune, val_loader, tokenizer, mode=f"Val FT {epoch}")

    # Checkpoint
    if val_rouge['rougeL'] > best_rouge:
        improvement = val_rouge['rougeL'] - best_rouge
        best_rouge = val_rouge['rougeL']

        torch.save(model_finetune.state_dict(), checkpoint_path)
        print(f"   Best model: ROUGE-L tăng: (+{improvement:.4f} -> {best_rouge:.4f})")
    else:
        print(f"   Không cải thiện (Best: {best_rouge:.4f}).")


model_finetune.eval()
sample = next(iter(val_loader))

# Cbi input
input_ids = sample['input_ids'][0].unsqueeze(0).to(device)
mask = sample['attention_mask'][0].unsqueeze(0).to(device)
dep = sample['dep_matrix'][0].unsqueeze(0).to(device)

speaker_ids = sample.get('speaker_ids')
turn_ids = sample.get('turn_ids')
speaker_ids = speaker_ids[0].unsqueeze(0).to(device)
turn_ids = turn_ids[0].unsqueeze(0).to(device)

# Generate
gen_ids = model_finetune.generate(
    input_ids=input_ids,
    attention_mask=mask,
    dep_matrix=dep,
    speaker_ids=speaker_ids,
    turn_ids=turn_ids,
    max_length=100,
    no_repeat_ngram_size=3,
    repetition_penalty=1.2,
    decoder_start_token_id=tokenizer.bos_token_id
)
print("Summary:", tokenizer.decode(gen_ids[0], skip_special_tokens=True))


 Load best model: best_graph_bart_dialogsum.pt
Loading Backbone facebook/bart-base...


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Có path

 Evaluate baseline của model vừa load:
Đang đánh giá (Baseline)...


Evaluating:   0%|          | 0/52 [00:00<?, ?it/s]

   📉 Baseline Loss: 1.6535
   📝 ROUGE-1: 0.5082 | ROUGE-L: 0.4209
      Started point: (Best ROUGE-L cũ): 0.4209

 Fine-tune 3 epochs

--- Fine-tune Epoch 1/3 ---


...

   📉 Val FT 2 Loss: 1.6202
   📝 ROUGE-1: 0.5129 | ROUGE-L: 0.4249
   Không cải thiện (Best: 0.4254).

--- Fine-tune Epoch 3/3 ---


Training FT 3:   0%|          | 0/921 [00:00<?, ?it/s]

   Train Loss: 1.1946
Đang đánh giá (Val FT 3)...


Evaluating:   0%|          | 0/52 [00:00<?, ?it/s]

   📉 Val FT 3 Loss: 1.6231
   📝 ROUGE-1: 0.5126 | ROUGE-L: 0.4249
   Không cải thiện (Best: 0.4254).
Summary: A wants to get a puppy for her son. She took him to the animal shelter last Monday. He wanted to name it Lemmy.


In [ ]:
if 'model_finetune' not in locals():
    print("Chưa có model để finetune")
else:
    print("\n" + "="*60)
    print(f"Kết quả Fine-Tunning")
    print("="*60)

    model_finetune.eval()
    random_indices = random.sample(range(len(val_ds)), 10)

    with torch.no_grad():
        for i, idx in enumerate(random_indices):
            sample = val_ds[idx]
            input_ids = sample['input_ids'].unsqueeze(0).to(device)
            mask = sample['attention_mask'].unsqueeze(0).to(device)
            dep = sample['dep_matrix'].unsqueeze(0).to(device)


            speaker_ids = sample.get('speaker_ids')
            turn_ids = sample.get('turn_ids')
            speaker_ids = speaker_ids.unsqueeze(0).to(device)
            turn_ids = turn_ids.unsqueeze(0).to(device)

            # Lấy ground truth
            label_ids = sample['labels'].clone()
            label_ids[label_ids == -100] = tokenizer.pad_token_id
            ground_truth = tokenizer.decode(label_ids, skip_special_tokens=True)

            # Lấy input dialogue
            dialogue_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

            # Generate context của model
            gen_ids = model_finetune.generate(
                input_ids=input_ids,
                attention_mask=mask,
                dep_matrix=dep,
                speaker_ids=speaker_ids,
                turn_ids=turn_ids,
                max_length=128,
                num_beams=4,              # Beam Search giúp câu văn chau chuốt hơn
                early_stopping=True,
                no_repeat_ngram_size=3,   # Cấm lặp cụm 3 từ
                repetition_penalty=1.2,   # Phạt lặp từ đơn
                length_penalty=1.0,
                decoder_start_token_id=tokenizer.bos_token_id
            )
            generated_summary = tokenizer.decode(gen_ids[0], skip_special_tokens=True)


            print(f"\n SAMPLE {i+1} (Index {idx}):")
            print(f"🗣️  DIALOGUE: {dialogue_text[:200]}...")
            print(f"🎯 GROUND TRUTH: {ground_truth}")
            print(f"🤖 FINE-TUNED PREDICT: {generated_summary}")
            print("-" * 60)


Kết quả Fine-Tunning

 SAMPLE 1 (Index 758):
🗣️  DIALOGUE: Mia : soup today ? 
 Mira : but no tomato soup please 
 Mia : ok , so maybe onion soup 
 Mira : sounds better :)...
🎯 GROUND TRUTH: Mira and Mia agree on onion soup today.
🤖 FINE-TUNED PREDICT: Mia and Mira will have onion soup today.
------------------------------------------------------------

 SAMPLE 2 (Index 558):
🗣️  DIALOGUE: Jody : I hate ironing ! 
 Lynn : Me too ! 
 Jody : Especially shirts ! 
 Lynn : I know , that 's a pain in the neck ! 
 Jody : My hubby uses like 5 shirts a week ! And all I do is iron them ! 
 Lynn :...
🎯 GROUND TRUTH: Jody hates ironing, especially her husband's shirts. She irons them because last time he did it, he burnt her favourite shirt and two of his own. Lynn suggests buying an ironing machine. Jody will visit her tomorrow around 2:00 to see how it works.
🤖 FINE-TUNED PREDICT: Jody hates ironing. Her hubby burnt her favourite shirts and two of his own. Jody has an ironing machine at home. S